In [1]:
import os
import zipfile
import urllib.request

# Path jahan dataset download aur extract hoga
dataset_dir = './dataset'
zip_path = os.path.join(dataset_dir, 'plantvillage.zip')

if not os.path.exists(dataset_dir):
    os.makedirs(dataset_dir)

# NAYA WORKING LINK (PlantVillage dataset ka stable mirror)
url = "https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip"

print("Dataset download ho raha hai... isme thoda time lag sakta hai.")
try:
    if not os.path.exists(zip_path):
        # User-Agent header add kar rahe hain taake server block na kare
        opener = urllib.request.build_opener()
        opener.addheaders = [('User-agent', 'Mozilla/5.0')]
        urllib.request.install_opener(opener)
        
        urllib.request.urlretrieve(url, zip_path)
        print("Download Complete!")
    else:
        print("Zip file pehle se maujood hai.")

    # Unzip karna
    print("Extracting files... Please wait...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print("Dataset successfully extract ho gaya hai aapke 'dataset' folder mein!")

except Exception as e:
    print(f"Koi masla aaya: {e}")

Dataset download ho raha hai... isme thoda time lag sakta hai.
Zip file pehle se maujood hai.
Extracting files... Please wait...
Dataset successfully extract ho gaya hai aapke 'dataset' folder mein!


In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# RAM crash se bachne ke liye size aur batch size kam kar rahe hain
IMAGE_SIZE = 128  # 224 se badal kar 128 kar diya
BATCH_SIZE = 16   # 32 se badal kar 16 kar diya

# Path bilkul sahi hai aapka
DATASET_PATH = "./dataset/PlantVillage-Dataset-master/raw/color" 

# Data Augmentation & Rescaling (Optimized for performance)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2 # 20% validation data
)

# Training Data Loader
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

# Validation Data Loader
validation_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print(f"Total Classes Found: {train_generator.num_classes}")

Found 43456 images belonging to 38 classes.
Found 10849 images belonging to 38 classes.
Total Classes Found: 38


In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

# 1. CNN Model Design Karte Hain
model = Sequential()

# Block 1 -> YAHAN 224 KO BADAL KAR 128 KIYA HAI
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# Block 2
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# Block 3
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# Flattening (Images ko 1D vector mein convert karna)
model.add(Flatten())

# Dense Layers (Fully Connected)
model.add(Dense(512, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5)) # Overfitting se bachne ke liye strong dropout

# Output Layer (38 Classes hain hamare dataset mein)
model.add(Dense(38, activation='softmax'))

# 2. Model Ko Compile Karte Hain
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Model ki summary print karte hain structure dekhne ke liye
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │    12,845,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 38)             │        19,494 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,961,254 (49.44 MB)

 Trainable params: 12,959,782 (49.44 MB)

 Non-trainable params: 1,472 (5.75 KB)

In [4]:
import tensorflow as tf

# 1. Checkpoint setup
checkpoint_path = "./models/best_plant_model.h5"
checkpoint = tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')

# 2. Steps calculations
spe = train_generator.samples // train_generator.batch_size
vs = validation_generator.samples // validation_generator.batch_size

print("Training setup locked! Starting epochs...")

# 3. Training Loop (Simple Line)
history = model.fit(train_generator, steps_per_epoch=spe, validation_data=validation_generator, validation_steps=vs, epochs=5, callbacks=[checkpoint])

Training setup locked! Starting epochs...
Epoch 1/5
2716/2716 ━━━━━━━━━━━━━━━━━━━━ 0s 621ms/step - accuracy: 0.5088 - loss: 1.8771
Epoch 1: val_accuracy improved from None to 0.56739, saving model to ./models/best_plant_model.h5



Epoch 1: finished saving model to ./models/best_plant_model.h5
2716/2716 ━━━━━━━━━━━━━━━━━━━━ 1856s 682ms/step - accuracy: 0.6064 - loss: 1.4051 - val_accuracy: 0.5674 - val_loss: 1.6941
Epoch 2/5
2716/2716 ━━━━━━━━━━━━━━━━━━━━ 0s 551ms/step - accuracy: 0.7145 - loss: 0.9540
Epoch 2: val_accuracy improved from 0.56739 to 0.58527, saving model to ./models/best_plant_model.h5



Epoch 2: finished saving model to ./models/best_plant_model.h5
2716/2716 ━━━━━━━━━━━━━━━━━━━━ 1587s 584ms/step - accuracy: 0.7305 - loss: 0.8910 - val_accuracy: 0.5853 - val_loss: 1.6392
Epoch 3/5
2716/2716 ━━━━━━━━━━━━━━━━━━━━ 0s 545ms/step - accuracy: 0.7851 - loss: 0.6908
Epoch 3: val_accuracy improved from 0.58527 to 0.86781, saving model to ./models/best_plant_model.h5



Epoch 3: finished saving model to ./models/best_plant_model.h5
2716/2716 ━━━━━━━━━━━━━━━━━━━━ 1581s 582ms/step - accuracy: 0.7959 - loss: 0.6524 - val_accuracy: 0.8678 - val_loss: 0.4179
Epoch 4/5
2716/2716 ━━━━━━━━━━━━━━━━━━━━ 0s 577ms/step - accuracy: 0.8361 - loss: 0.5201
Epoch 4: val_accuracy did not improve from 0.86781
2716/2716 ━━━━━━━━━━━━━━━━━━━━ 1651s 608ms/step - accuracy: 0.8386 - loss: 0.5122 - val_accuracy: 0.7859 - val_loss: 0.8527
Epoch 5/5
2716/2716 ━━━━━━━━━━━━━━━━━━━━ 0s 536ms/step - accuracy: 0.8617 - loss: 0.4252
Epoch 5: val_accuracy improved from 0.86781 to 0.89113, saving model to ./models/best_plant_model.h5



Epoch 5: finished saving model to ./models/best_plant_model.h5
2716/2716 ━━━━━━━━━━━━━━━━━━━━ 1567s 577ms/step - accuracy: 0.8588 - loss: 0.4424 - val_accuracy: 0.8911 - val_loss: 0.3363
